# Lesson 01 - एआई एजेंट्स का परिचय

**AI Agents for Beginners** कोर्स के पहले पाठ में आपका स्वागत है!

एक **AI एजेंट** एक प्रोग्राम है जो एक बड़े भाषा मॉडल (LLM) का उपयोग अपने तर्क इंजन के रूप में करता है और वास्तविक दुनिया में *कार्रवाइयां* कर सकता है — API कॉल करना, डेटाबेस से प्रश्न पूछना, या कोड चलाना — ताकि उपयोगकर्ता की ओर से कोई लक्ष्य पूरा किया जा सके।

इस नोटबुक में आप अपना पहला एजेंट बनाएंगे: एक **ट्रैवल एजेंट** जो छुट्टियों के लिए गंतव्य की सिफारिश करता है। इस बीच आप सीखेंगे कि कैसे:

1. **Microsoft Agent Framework** का उपयोग करके Azure AI Foundry Agent Service से कनेक्ट किया जाता है।
2. एजेंट को एक **टूल** दें — एक साधारण Python फ़ंक्शन जिसे वह कॉल कर सकता है।
3. एजेंट चलाएँ और उसके उत्तर का निरीक्षण करें।
4. एजेंट के उत्तर को टोकन-दर-टोकन स्ट्रीम करें।


## सेटअप

इस नोटबुक को चलाने से पहले, सुनिश्चित करें कि आपके पास है:

1. **एक Azure AI Foundry प्रोजेक्ट** जिसमें एक तैनात चैट मॉडल (जैसे `gpt-4o-mini`) हो।
2. **Azure CLI से लॉगिन किया हुआ है** — अपने टर्मिनल में `az login` चलाएं।
3. **आवश्यक एनवायरनमेंट वैरिएबल सेट किए हों:**
   - `AZURE_AI_PROJECT_ENDPOINT` — आपका Azure AI Foundry प्रोजेक्ट एंडपॉइंट।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — आपके तैनात मॉडल का नाम।

नीचे का सेल आपके लिए आवश्यक पायथन पैकेज इंस्टॉल करता है।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## अपना पहला एजेंट बनाना

एक एजेंट को दो चीजें चाहिए:

- **निर्देश** जो उसे *कौन है* और *कैसे व्यवहार करना है* बताते हैं (एक सिस्टम प्रॉम्प्ट)।
- **उपकरण** — Python फ़ंक्शन जिन्हें `@tool` से सजाया गया है जिन्हें एजेंट जानकारी प्राप्त करने या कार्य करने के लिए कॉल कर सकता है।

नीचे हमने एक सरल उपकरण परिभाषित किया है जो लोकप्रिय छुट्टी स्थानों की एक सूची लौटाता है। जब उपयोगकर्ता यात्रा सुझाव मांगता है तो एजेंट इस उपकरण का उपयोग करेगा।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रीमिंग प्रतिक्रियाएँ

एक अधिक संवादात्मक अनुभव के लिए आप एजेंट की प्रतिक्रिया को **स्ट्रीम** कर सकते हैं। पूर्ण उत्तर की प्रतीक्षा करने के बजाय, एजेंट टेक्स्ट के टुकड़े उत्पन्न होने पर ही देता है। यह विशेष रूप से चैट इंटरफेस में उपयोगी होता है जहाँ आप आउटपुट को रीयल टाइम में प्रदर्शित करना चाहते हैं।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

इस पाठ में आपने सीखा कि कैसे:

- **एक प्रदाता बनाएँ** जो `AzureAIProjectAgentProvider` के माध्यम से Azure AI Foundry Agent Service से कनेक्ट होता है।
- **एक उपकरण परिभाषित करें** `@tool` डेकोरेटर का उपयोग करके ताकि एजेंट आपकी Python फ़ंक्शन्स को कॉल कर सके।
- **एजेंट चलाएँ** एक उपयोगकर्ता संदेश के साथ और उसकी प्रतिक्रिया प्रिंट करें।
- **प्रतिक्रियाओं को स्ट्रीम करें** रियल-टाइम आउटपुट के लिए।

अगले पाठ में हम एजेंटिक फ्रेमवर्क्स को गहराई से जानेंगे और सीखेंगे कि कैसे एजेंट्स को अधिक शक्तिशाली उपकरण और बहु-चरण तर्क क्षमताएँ दी जाएं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। हम सटीकता के लिए प्रयासरत हैं, लेकिन कृपया ध्यान दें कि स्वचालित अनुवाद में त्रुटियाँ या असत्यताएँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही अधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
